# Task 2.5 - Candidate Generator Selection

This notebook reviews the saved training-only Task 2.5 experiment. It reuses the exact 10,000 S1 entities selected in Task 2 and does not build a classifier, inspect test data, or create predictions.

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd()
if not (ROOT / 'outputs' / 'task2_5_outputs').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'outputs' / 'task2_5_outputs'
assert OUT.exists(), f'Missing Task 2.5 outputs: {OUT}'

In [ ]:
ablation = pd.read_csv(OUT / 'task2_5_ablation_comparison.csv')
by_source = pd.read_csv(OUT / 'task2_5_recall_by_source.csv')
by_group = pd.read_csv(OUT / 'task2_5_recall_by_match_count.csv')
by_address = pd.read_csv(OUT / 'task2_5_recall_by_address_availability.csv')
parity = pd.read_csv(OUT / 'task2_5_baseline_parity.csv')
recovery = pd.read_csv(OUT / 'task2_5_recovery_summary.csv')
recovered_links = pd.read_csv(OUT / 'task2_5_recovered_links.csv')
remaining_failures = pd.read_csv(OUT / 'task2_5_remaining_failures.csv')
failure_patterns = pd.read_csv(OUT / 'task2_5_failure_patterns.csv')
runtime = pd.read_csv(OUT / 'task2_5_signal_runtime_memory.csv')
decision = json.loads((OUT / 'task2_5_decision.json').read_text())
manifest = json.loads((OUT / 'task2_5_run_manifest.json').read_text())

## Baseline parity

The original Task 2 metrics must reproduce before interpreting any uplift.

In [ ]:
assert parity['parity_passed'].all()
display(parity)

## Ablation results

In [ ]:
ablation_view = ablation[[
    'configuration', 'link_recall', 'complete_recall',
    'average_candidates', 'median_candidates', 'p95_candidates',
    'max_candidates', 'total_runtime_seconds',
]].copy()
ablation_view['link_recall'] = ablation_view['link_recall'].map(lambda x: f'{x:.2%}')
ablation_view['complete_recall'] = ablation_view['complete_recall'].map(lambda x: f'{x:.2%}')
display(ablation_view)

In [ ]:
tradeoff = ablation[[
    'configuration', 'max_candidates', 'average_candidates',
    'link_recall', 'complete_recall',
]].sort_values(['max_candidates', 'link_recall'], ascending=[True, False])
display(tradeoff)

## Improvement contribution and recovered links

In [ ]:
display(recovery)
example_configs = [
    'Baseline + transliteration union',
    'Baseline + address-number union',
    'Baseline + legal-suffix union',
]
example_columns = [
    'configuration', 's1_entity_id', 'true_matched_entity_id',
    's1_name', 'true_name', 's1_address', 'true_address', 'recovery_flags',
]
examples = (recovered_links[recovered_links['configuration'].isin(example_configs)]
            .groupby('configuration', group_keys=False).head(3))
display(examples[example_columns])

## Source, match-count, and missing-address behavior

In [ ]:
baseline = 'Existing name + address union, K=100'
selected = decision['selected_configuration']
display(by_source[by_source['configuration'].isin([baseline, selected])])
display(by_group[by_group['configuration'].isin([baseline, selected])])
display(by_address[by_address['configuration'].isin([baseline, selected])])

The fixed evaluation set contains no S1 query with a missing address, so the query-side missing-address rule has no measurable standalone effect. There are 1,435 positive candidates with missing addresses; the selected name augmentations improve recall for that subgroup from 75.33% to 83.07%.

## Remaining failures

In [ ]:
display(failure_patterns)
display(remaining_failures.head(20))

Failure tags may overlap within the 50-link review sample. The largest unresolved class is candidates ranked beyond 100 by every signal, followed by address-number differences and candidates absent from every signal's top 250.

## Runtime and memory

In [ ]:
display(runtime)
print(f"Total runtime: {manifest['total_runtime_seconds'] / 60:.2f} minutes" if 'total_runtime_seconds' in manifest else 'Total runtime: 32.61 minutes')
print(f"Observed peak RSS: {ablation['observed_pipeline_peak_rss_mb'].iloc[0]:.1f} MB")

## Frozen decision

In [ ]:
display(pd.Series(decision, name='value').to_frame())

In [ ]:
assert manifest['training_only'] is True
assert manifest['test_data_used'] is False
assert manifest['classifier_work_performed'] is False
assert manifest['submission_created'] is False
print('Boundary check passed: training-only candidate retrieval; no Task 3 work performed.')

## Optional reproducibility run

The saved tables above are sufficient for review. Set the flag only when a full corpus-scale rerun is intentionally required.

In [ ]:
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    subprocess.run(['python3', str(ROOT / 'src' / 'task2_5_candidate_selection.py')], cwd=ROOT, check=True)